# Deep Learning 045 — CNN Architecture and LeNet-5

Every CNN is one skeleton: **image → (conv → activation → pool) × N → flatten →
dense → output**. LeNet-5 (LeCun, 1998) is the first instance, with **60,806**
parameters — and **97.3%** of them sit in the dense head.

| Part | What we check |
|---|---|
| A | count LeNet-5 layer by layer |
| B | the sparse C3 table against modern full connectivity |
| C | where the parameters actually are |
| D | the shape of the skeleton |

In [ ]:
import numpy as np

## Part A — Counting LeNet-5

In [ ]:
def conv_params(filters, k, in_ch):
    return filters * (k * k * in_ch) + filters

def dense_params(units, inputs):
    return units * inputs + units

layers = [
    ("C1  conv 6 x 5x5x1",   conv_params(6, 5, 1)),
    ("S2  pool",             0),
    ("C3  conv 16 x 5x5 (sparse table)", 1516),
    ("S4  pool",             0),
    ("C5  conv 120 x 5x5x16", conv_params(120, 5, 16)),
    ("F6  dense 84",          dense_params(84, 120)),
    ("out dense 10",          dense_params(10, 84)),
]
total = 0
for name, p in layers:
    total += p
    print(f"{name:<36}{p:>8,}")
print(f"{'total':<36}{total:>8,}")
assert total == 60_806

## Part B — The sparse table

C3 in the original paper does not connect every input map to every filter; it
uses a hand-designed table. Modern implementations connect everything.

In [ ]:
full_c3 = conv_params(16, 5, 6)
print(f"C3 with the 1998 sparse table : 1,516")
print(f"C3 fully connected            : {full_c3:,}")
print(f"total, modern version         : {60_806 - 1516 + full_c3:,}")
assert full_c3 == 2416 and 60_806 - 1516 + full_c3 == 61_706
print("\nThe table saved 900 parameters and some computation, on 1998 hardware.")

## Part C — Where the parameters live

In [ ]:
conv_side = 156 + 1516 + 48120          # C1, C3, C5
dense_side = 10164 + 850                # F6, output
# C5 is a convolution by construction but is fully connected in practice (5x5 on a 5x5 map)
head = 48120 + 10164 + 850
print(f"feature extraction (C1, C3)      {156 + 1516:>8,}  {(156+1516)/60806:>6.1%}")
print(f"classifier head (C5, F6, output) {head:>8,}  {head/60806:>6.1%}")
assert round(head / 60806 * 100, 1) == 97.3
print("\nThe same pattern as every other lesson in this course: the layer facing")
print("a large flat space holds the parameters; the interesting layers are cheap.")

## Part D — The skeleton

In [ ]:
skeleton = ["image", "conv", "activation", "pool", "conv", "activation", "pool",
            "flatten", "dense", "output"]
print(" -> ".join(skeleton))
print("\nBefore the flatten: feature extraction. After it: an ordinary ANN.")
print("That split is exactly what transfer learning reuses (lesson 053).")

## What to take away

- **One skeleton:** image → (conv → activation → pool) × N → flatten → dense → output.
- **LeNet-5: 60,806 parameters**, five weighted layers, 1998.
- Its **C3 sparse table gives 1,516**; full connectivity gives **2,416** and a
  total of **61,706**.
- **97.3% of the parameters are in the dense head.**

## Exercises

1. Add a second dense layer of 200 units before the output. How much does the
   head grow?
2. Double every filter count. Which side grows faster, extraction or head?
3. Work out C5's parameter count if the input map were 7×7 instead of 5×5.